In [3]:
from tdc.multi_pred import DDI

raw_data_db = DDI(name="DrugBank", path="/media/onlyzabao/Data/TDC").get_data()
print(raw_data_db.shape)

Found local copy...
Loading...
Done!


(191808, 5)


In [4]:
import pandas as pd
import json

# Filter row
selectedLabel = [31, 49, 50, 56, 66, 86]
data_db = raw_data_db[raw_data_db["Y"].isin(selectedLabel)]

# Filter column
data_db = data_db[["Drug1_ID", "Drug2_ID", "Y"]]

# Filter hetionet
with open("../../Hetionet/vocab/entity_vocab.json", "r") as file:
    entity = json.load(file)

data_db["Drug1_ID"] = "Compound::" + data_db["Drug1_ID"]
data_db["Drug2_ID"] = "Compound::" + data_db["Drug2_ID"]
data_db = data_db[
    data_db["Drug1_ID"].isin(entity)
    & data_db["Drug2_ID"].isin(entity)
]

data_db["Interaction"] = "CaC"

print(data_db)

                 Drug1_ID           Drug2_ID   Y Interaction
28691   Compound::DB00968  Compound::DB00752  31         CaC
28692   Compound::DB00968  Compound::DB01037  31         CaC
28693   Compound::DB00968  Compound::DB01626  31         CaC
28694   Compound::DB00968  Compound::DB00614  31         CaC
28695   Compound::DB00968  Compound::DB01171  31         CaC
...                   ...                ...  ..         ...
191802  Compound::DB00437  Compound::DB00542  86         CaC
191803  Compound::DB00437  Compound::DB00492  86         CaC
191805  Compound::DB00437  Compound::DB00790  86         CaC
191806  Compound::DB00415  Compound::DB00437  86         CaC
191807  Compound::DB00437  Compound::DB00691  86         CaC

[46667 rows x 4 columns]


In [5]:
raw_data_ts = pd.read_csv("/media/onlyzabao/Data/TDC/final.csv", delimiter=",")
print(raw_data_ts.shape)

(1808341, 8)


In [10]:
# Filter columns
data_ts = raw_data_ts.loc[
    :, ["DrugBank ID", "DrugBank ID_ID2", "Y", "Side Effect Name"]
]
data_ts = data_ts.rename(
    columns={
        "DrugBank ID": "Drug1_ID",
        "DrugBank ID_ID2": "Drug2_ID",
        "Side Effect Name": "SideEffect",
    }
)

# Filter rows

with open("../../Hetionet/vocab/entity_vocab.json", "r") as file:
    entity = json.load(file)

data_ts["Drug1_ID"] = "Compound::" + data_ts["Drug1_ID"]
data_ts["Drug2_ID"] = "Compound::" + data_ts["Drug2_ID"]
data_ts = data_ts[data_ts["Drug1_ID"].isin(entity) & data_ts["Drug2_ID"].isin(entity)]

data_ts["Interaction"] = "CaC"

print(data_ts)

                  Drug1_ID           Drug2_ID     Y  \
0        Compound::DB01236  Compound::DB01223   853   
1        Compound::DB01236  Compound::DB01223   841   
2        Compound::DB01236  Compound::DB01223   830   
3        Compound::DB01236  Compound::DB01223  1236   
4        Compound::DB01236  Compound::DB01223  1221   
...                    ...                ...   ...   
1808063  Compound::DB01019  Compound::DB00996  1043   
1808064  Compound::DB01019  Compound::DB00996  1115   
1808065  Compound::DB01019  Compound::DB00996   529   
1808066  Compound::DB01019  Compound::DB00996   893   
1808067  Compound::DB01019  Compound::DB00996   836   

                                 SideEffect Interaction  
0                          thrombocytopenia         CaC  
1                         sinus tachycardia         CaC  
2                                   stridor         CaC  
3           arterial pressure NOS increased         CaC  
4                           Hypoventilation      

In [12]:
from sklearn.model_selection import train_test_split


# def split_data(data: pd.DataFrame, train_size, test_size, val_size):
#     train_data, test_data = train_test_split(data, test_size=1 - train_size)
#     test_data, val_data = train_test_split(
#         test_data, test_size=val_size / (test_size + val_size)
#     )

#     return train_data, test_data, val_data


# train_data, test_data, val_data = split_data(data, 0.7, 0.2, 0.1)

# print(train_data.shape)

test_data, val_data = train_test_split(data_ts, test_size=0.1)

print(test_data.shape)
print(val_data.shape)

(1393261, 5)
(154807, 5)


In [13]:
def write_data(path, data: pd.DataFrame, head_index, tail_index, edge_index, inverses):
    with open(path, "w") as file:
        for _, row in data.iterrows():
            if not inverses:
                head = row[head_index]
                tail = row[tail_index]
                edge = row[edge_index]
            else:
                head = row[tail_index]
                tail = row[head_index]
                edge = "_" + row[edge_index]

            file.write(head + "\t" + edge + "\t" + tail + "\n")


files = [
    {"path": "../dev.txt", "data": val_data, "inverses": False},
    {"path": "../test.txt", "data": test_data, "inverses": False},
    {"path": "../train.txt", "data": data_db, "inverses": False},
    {"path": "../graph_triples_DB.txt", "data": data_db, "inverses": False},
    {"path": "../graph_inverses_DB.txt", "data": data_db, "inverses": True},
]

head_index = "Drug1_ID"
tail_index = "Drug2_ID"
edge_index = "Interaction"
for file in files:
    write_data(
        file["path"], file["data"], head_index, tail_index, edge_index, file["inverses"]
    )